In [0]:
%sql
-- 1. Creación del Schema Silver
CREATE SCHEMA IF NOT EXISTS workspace.silver;

-- 2. Tabla de Auditoría / Cuarentena de Data Quality
CREATE OR REPLACE TABLE workspace.silver.sales_order_quarantine (
    QuarantineKey BIGINT GENERATED ALWAYS AS IDENTITY,
    SourceEntity STRING NOT NULL,         -- 'HEADER' o 'ITEM'
    RecordIdentifier STRING,              -- SalesOrder o SalesOrder + SalesOrderItem
    FailedRule STRING NOT NULL,           -- e.g. 'RULE_1_NULL_PK', 'RULE_8_QUADRATURE_MISMATCH'
    FailureSeverity STRING NOT NULL,     -- 'HARD' o 'SOFT'
    RawRecord STRING NOT NULL,            -- Snapshot crudo del JSON del payload
    IngestionTimestamp TIMESTAMP NOT NULL
)
CLUSTER BY (SourceEntity, FailedRule);

-- 3. Dimensión Fecha (Calendario)
CREATE OR REPLACE TABLE workspace.silver.dim_date (
    DateKey INT NOT NULL,                 -- Formato YYYYMMDD
    FullDate DATE NOT NULL,
    YearNbr INT NOT NULL,
    QuarterNbr INT NOT NULL,
    MonthNbr INT NOT NULL,
    MonthName STRING NOT NULL,
    DayNbr INT NOT NULL,
    DayOfWeekName STRING NOT NULL,
    CONSTRAINT pk_dim_date PRIMARY KEY (DateKey)
)
CLUSTER BY (DateKey);

-- 4. Dimensión Cliente (SCD Tipo 2)
CREATE OR REPLACE TABLE workspace.silver.dim_customer (
    CustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,
    SoldToParty STRING NOT NULL,
    CustomerGroup STRING,
    CustomerAccountAssignmentGroup STRING,
    CustomerPaymentTerms STRING,
    -- Columnas de Control Histórico SCD Tipo 2
    ValidFrom TIMESTAMP NOT NULL,
    ValidTo TIMESTAMP,
    IsCurrent BOOLEAN NOT NULL,
    CONSTRAINT pk_dim_customer PRIMARY KEY (CustomerKey)
)
CLUSTER BY (SoldToParty);

-- 5. Dimensión Producto (SCD Tipo 2)
CREATE OR REPLACE TABLE workspace.silver.dim_product (
    ProductKey BIGINT GENERATED ALWAYS AS IDENTITY,
    Material STRING NOT NULL,
    MaterialByCustomer STRING,
    MaterialGroup STRING,
    MaterialPricingGroup STRING,
    OriginallyRequestedMaterial STRING,
    -- Columnas de Control Histórico SCD Tipo 2
    ValidFrom TIMESTAMP NOT NULL,
    ValidTo TIMESTAMP,
    IsCurrent BOOLEAN NOT NULL,
    CONSTRAINT pk_dim_product PRIMARY KEY (ProductKey)
)
CLUSTER BY (Material);

-- 6. Actualización de FactSalesOrderItem
CREATE OR REPLACE TABLE workspace.silver.fact_sales_order_item (
    FactSalesOrderItemKey BIGINT GENERATED ALWAYS AS IDENTITY,
    -- Claves de Negocio / Degeneradas
    SalesOrder STRING NOT NULL,
    SalesOrderItem STRING NOT NULL,
    -- Claves Foráneas hacia Dimensiones
    CustomerKey BIGINT,
    ProductKey BIGINT,
    CreationDateKey INT,
    SalesOrderDateKey INT,
    RequestedDeliveryDateKey INT,
    -- Atributos Descriptivos y de Estado
    SalesOrderItemCategory STRING,
    DeliveryStatus STRING,
    SDProcessStatus STRING,
    BillingStatus STRING,
    TransactionCurrency STRING,
    -- Métricas Transaccionales
    RequestedQuantity DECIMAL(18, 3),
    ConfdDeliveredQuantity DECIMAL(18, 3),
    NetAmount DECIMAL(18, 2),
    TaxAmount DECIMAL(18, 2),
    CostAmount DECIMAL(18, 2),
    GrossWeight DECIMAL(18, 3),
    NetWeight DECIMAL(18, 3),
    ItemVolume DECIMAL(18, 3),
    -- Flags de Calidad y Auditoría
    IsDeleted BOOLEAN,
    DqWarnings ARRAY<STRING>,
    _ingestion_timestamp TIMESTAMP,
    -- Restricciones Relacionales (Unity Catalog)
    CONSTRAINT pk_fact_sales PRIMARY KEY (FactSalesOrderItemKey),
    CONSTRAINT fk_fact_customer FOREIGN KEY (CustomerKey) REFERENCES workspace.silver.dim_customer(CustomerKey),
    CONSTRAINT fk_fact_product FOREIGN KEY (ProductKey) REFERENCES workspace.silver.dim_product(ProductKey),
    CONSTRAINT fk_fact_creation_date FOREIGN KEY (CreationDateKey) REFERENCES workspace.silver.dim_date(DateKey),
    CONSTRAINT fk_fact_order_date FOREIGN KEY (SalesOrderDateKey) REFERENCES workspace.silver.dim_date(DateKey),
    CONSTRAINT fk_fact_delivery_date FOREIGN KEY (RequestedDeliveryDateKey) REFERENCES workspace.silver.dim_date(DateKey)
)
CLUSTER BY (SalesOrder, CreationDateKey);